In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("data/messy_data.csv")

In [7]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


In [35]:
print("Column names after standardizing:")
print(df.columns.tolist())

Column names after standardizing:
['patient_name', 'age', 'gender', 'condition', 'medication', 'visit_date', 'cholesterol', 'systolic', 'diastolic', 'visit_year', 'visit_month']


In [8]:
df.replace(["Unknown", "", " "], np.nan, inplace=True)


In [34]:
print("Missing values per column after replacing placeholders:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

Missing values per column after replacing placeholders:
patient_name    0
age             0
gender          0
condition       0
medication      0
visit_date      0
cholesterol     0
systolic        0
diastolic       0
visit_year      0
visit_month     0
dtype: int64

Total missing values: 0


In [25]:
# Fix 1: convert word-numbers to digits before coercing
word_to_num = {
    'zero':0,'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,
    'eight':8,'nine':9,'ten':10,'eleven':11,'twelve':12,'thirteen':13,
    'fourteen':14,'fifteen':15,'sixteen':16,'seventeen':17,'eighteen':18,
    'nineteen':19,'twenty':20,'twenty-one':21,'twenty-two':22,'twenty-three':23,
    'twenty-four':24,'twenty-five':25,'twenty-six':26,'twenty-seven':27,
    'twenty-eight':28,'twenty-nine':29,'thirty':30,'thirty-one':31,
    'thirty-two':32,'thirty-three':33,'thirty-four':34,'thirty-five':35,
    'thirty-six':36,'thirty-seven':37,'thirty-eight':38,'thirty-nine':39,
    'forty':40,'forty-one':41,'forty-two':42,'forty-three':43,'forty-four':44,
    'forty-five':45,'forty-six':46,'forty-seven':47,'forty-eight':48,'forty-nine':49,
    'fifty':50,'fifty-one':51,'fifty-two':52,'fifty-three':53,'fifty-four':54,
    'fifty-five':55,'fifty-six':56,'fifty-seven':57,'fifty-eight':58,'fifty-nine':59,
    'sixty':60,'sixty-one':61,'sixty-two':62,'sixty-three':63,'sixty-four':64,
    'sixty-five':65,'sixty-six':66,'sixty-seven':67,'sixty-eight':68,'sixty-nine':69,
    'seventy':70,'seventy-five':75,'eighty':80,'eighty-five':85,'ninety':90,
}
df['age'] = (
    df['age'].astype(str).str.strip().str.lower()
    .map(lambda x: str(word_to_num.get(x, x)))
)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].astype('Int64')

df['visit_date'] = pd.to_datetime(df['visit_date'], format='mixed', dayfirst=False, errors='coerce')


In [33]:
print("Age column after cleaning:")
print(df['age'].describe())
print("\nMissing values remaining:", df['age'].isnull().sum())
print("Data type:", df['age'].dtype)
print("Sample values:", df['age'].head(10).tolist())

Age column after cleaning:
count        900.0
mean     48.322222
std      13.048918
min           18.0
25%           41.0
50%           49.0
75%           57.0
max           85.0
Name: age, dtype: Float64

Missing values remaining: 0
Data type: Int64
Sample values: [51, 49, 49, 46, 49, 66, 62, 61, 49, 68]


In [26]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_name  900 non-null    object        
 1   age           900 non-null    Int64         
 2   gender        900 non-null    object        
 3   condition     900 non-null    object        
 4   medication    900 non-null    object        
 5   visit_date    900 non-null    datetime64[ns]
 6   cholesterol   900 non-null    float64       
 7   systolic      900 non-null    float64       
 8   diastolic     900 non-null    float64       
 9   visit_year    900 non-null    int32         
 10  visit_month   900 non-null    int32         
dtypes: Int64(1), datetime64[ns](1), float64(3), int32(2), object(4)
memory usage: 71.3+ KB
None


In [27]:
df['gender'] = df['gender'].str.strip().str.lower()

# Fix 3: normalise gender abbreviations and variants
gender_map = {
    'm': 'male', 'man': 'male',
    'f': 'female', 'woman': 'female',
    'non-binary': 'other',
}
df['gender'] = df['gender'].replace(gender_map)



In [32]:
print("Gender value counts after cleaning:")
print(df['gender'].value_counts())
print("\nMissing values remaining:", df['gender'].isnull().sum())
print("Unique values:", df['gender'].unique())

Gender value counts after cleaning:
gender
female    448
male      413
other      39
Name: count, dtype: int64

Missing values remaining: 0
Unique values: ['female' 'male' 'other']


In [28]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_name  900 non-null    object        
 1   age           900 non-null    Int64         
 2   gender        900 non-null    object        
 3   condition     900 non-null    object        
 4   medication    900 non-null    object        
 5   visit_date    900 non-null    datetime64[ns]
 6   cholesterol   900 non-null    float64       
 7   systolic      900 non-null    float64       
 8   diastolic     900 non-null    float64       
 9   visit_year    900 non-null    int32         
 10  visit_month   900 non-null    int32         
dtypes: Int64(1), datetime64[ns](1), float64(3), int32(2), object(4)
memory usage: 71.3+ KB
None


In [ ]:
df['medication'] = df['medication'].str.strip().str.upper()

In [29]:
# Fix 4: map all 'no medication' variants to NONE, fill genuinely missing with NONE
df['medication'] = df['medication'].replace({
    'NO_MEDICATION': 'NONE',
    'N/A': 'NONE',
    'NA': 'NONE',
})
df['medication'] = df['medication'].fillna('NONE')


In [30]:
print("Unique medication values:")
print(df['medication'].value_counts())
print("\nMissing values remaining:", df['medication'].isnull().sum())

Unique medication values:
medication
LISINOPRIL      242
NONE            224
ATORVASTATIN    217
METFORMIN       114
ALBUTEROL       103
Name: count, dtype: int64

Missing values remaining: 0


In [12]:

condition_map = {
    'Asthma':'Asthma',        'asthma':'Asthma',        'ASTHMA':'Asthma',       'Athsma':'Asthma',
    'Diabetes':'Diabetes',    'diabetes':'Diabetes',    'DIABETES':'Diabetes',   'Diabets':'Diabetes',
    'Heart Disease':'Heart Disease', 'heart disease':'Heart Disease',
    'Heart disease':'Heart Disease', 'Hearth Disease':'Heart Disease',
    'Hypertension':'Hypertension', 'hypertension':'Hypertension',
    'HYPERTENSION':'Hypertension',  'Hypertention':'Hypertension',
}
df['condition'] = df['condition'].str.strip().map(
    lambda x: condition_map.get(x, x) if isinstance(x, str) else x
)
df = df.dropna(subset=['condition'])


In [31]:
print("Condition value counts after cleaning:")
print(df['condition'].value_counts())
print("\nMissing values remaining:", df['condition'].isnull().sum())
print("Total rows remaining:", len(df))

Condition value counts after cleaning:
condition
Heart Disease    226
Hypertension     225
Asthma           225
Diabetes         224
Name: count, dtype: int64

Missing values remaining: 0
Total rows remaining: 900


In [13]:
df = df.drop_duplicates()


In [36]:
print("After removing duplicates:")
print("Remaining rows:", len(df))
print("Duplicate rows remaining:", df.duplicated().sum())

After removing duplicates:
Remaining rows: 900
Duplicate rows remaining: 0


In [14]:
# Fix 6: strip unit strings like '215 mg/dL', handle 'nan' string, fill missing
df['cholesterol'] = (
    df['cholesterol'].astype(str)
    .str.strip()
    .str.replace(r'[^\d.]', '', regex=True)  
    .replace({'': np.nan, 'nan': np.nan})     
)
df['cholesterol'] = pd.to_numeric(df['cholesterol'], errors='coerce')
df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())


In [37]:
print("Cholesterol column after cleaning:")
print(df['cholesterol'].describe())
print("\nMissing values remaining:", df['cholesterol'].isnull().sum())
print("Data type:", df['cholesterol'].dtype)
print("\nSample values:")
print(df['cholesterol'].head(10).tolist())

Cholesterol column after cleaning:
count    900.000000
mean     209.697444
std       27.803851
min      145.700000
25%      190.275000
50%      206.200000
75%      224.700000
max      320.000000
Name: cholesterol, dtype: float64

Missing values remaining: 0
Data type: float64

Sample values:
[207.7, 206.3, 207.5, 203.6, 199.5, 206.2, 240.9, 199.3, 187.1, 208.5]


In [15]:
df.drop(columns=["email", "phone_number"], inplace=True, errors="ignore")


In [38]:
print("Columns remaining after dropping PII:")
print(df.columns.tolist())
print("\nTotal columns:", len(df.columns))

Columns remaining after dropping PII:
['patient_name', 'age', 'gender', 'condition', 'medication', 'visit_date', 'cholesterol', 'systolic', 'diastolic', 'visit_year', 'visit_month']

Total columns: 11


In [16]:
df.reset_index(drop=True, inplace=True)


In [39]:
print("Index after reset:")
print(df.index)
print("\nFirst few rows with new index:")
print(df.head())

Index after reset:
RangeIndex(start=0, stop=900, step=1)

First few rows with new index:
      patient_name  age  gender     condition    medication visit_date  \
0       donna hall   51  female      Diabetes  ATORVASTATIN 2021-06-19   
1       mark young   49    male      Diabetes     METFORMIN 2020-04-06   
2       lisa young   49    male  Hypertension          NONE 2018-05-23   
3  daniel robinson   46    male        Asthma     ALBUTEROL 2022-10-10   
4    robert garcia   49  female  Hypertension          NONE 2018-05-10   

   cholesterol  systolic  diastolic  visit_year  visit_month  
0        207.7     143.0       90.0        2021            6  
1        206.3     109.0       86.0        2020            4  
2        207.5     153.0       95.0        2018            5  
3        203.6     127.0       79.0        2022           10  
4        199.5     161.0       93.0        2018            5  


In [17]:
print(df.info())
print(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   patient_name    900 non-null    object        
 1   age             900 non-null    Int64         
 2   gender          900 non-null    object        
 3   condition       900 non-null    object        
 4   medication      900 non-null    object        
 5   visit_date      900 non-null    datetime64[ns]
 6   cholesterol     900 non-null    float64       
 7   blood_pressure  825 non-null    object        
dtypes: Int64(1), datetime64[ns](1), float64(1), object(5)
memory usage: 57.3+ KB
None
      patient_name  age  gender     condition    medication visit_date  \
0       donna hall   51  female      Diabetes  ATORVASTATIN 2021-06-19   
1       mark young   49    male      Diabetes     METFORMIN 2020-04-06   
2       lisa young   49    male  Hypertension          NONE 2018-05

In [18]:
df

,patient_name,age,gender,condition,medication,visit_date,cholesterol,blood_pressure
0,donna hall,51,female,Diabetes,ATORVASTATIN,2021-06-19,207.7,143/90
1,mark young,49,male,Diabetes,METFORMIN,2020-04-06,206.3,109 / 86
2,lisa young,49,male,Hypertension,NONE,2018-05-23,207.5,153/95
3,daniel robinson,46,male,Asthma,ALBUTEROL,2022-10-10,203.6,127/79
4,robert garcia,49,female,Hypertension,NONE,2018-05-10,199.5,161/93
...,...,...,...,...,...,...,...,...
895,james wilson,58,male,Diabetes,ATORVASTATIN,2020-09-10,228.3,NaN
896,donna martinez,49,female,Heart Disease,LISINOPRIL,2023-03-28,222.7,153/88
897,emily jackson,68,male,Diabetes,ATORVASTATIN,2019-02-21,213.8,153/80
898,robert walker,19,male,Asthma,NONE,2023-04-22,170.0,121/70


In [19]:
# Fix 7: strip spaces around '/' so '120 / 80' splits cleanly
df['blood_pressure'] = df['blood_pressure'].astype(str).str.replace(r'\s*/\s*', '/', regex=True)

df['systolic']  = df['blood_pressure'].str.split('/').str[0]
df['diastolic'] = df['blood_pressure'].str.split('/').str[1]

df['systolic']  = pd.to_numeric(df['systolic'],  errors='coerce')
df['diastolic'] = pd.to_numeric(df['diastolic'], errors='coerce')

df['systolic']  = df['systolic'].fillna(df['systolic'].median())
df['diastolic'] = df['diastolic'].fillna(df['diastolic'].median())

df.drop(columns=['blood_pressure'], inplace=True)


In [40]:
print("Blood pressure after splitting:")
print(df[['systolic', 'diastolic']].describe())
print("\nMissing values:")
print("Systolic: ", df['systolic'].isnull().sum())
print("Diastolic:", df['diastolic'].isnull().sum())
print("\nSample values:")
print(df[['systolic', 'diastolic']].head(10))
print("\nColumns remaining:", df.columns.tolist())

Blood pressure after splitting:
         systolic   diastolic
count  900.000000  900.000000
mean   136.946667   86.846667
std     17.242930   10.437527
min     90.000000   61.000000
25%    124.000000   79.000000
50%    136.000000   87.000000
75%    149.000000   94.000000
max    194.000000  114.000000

Missing values:
Systolic:  0
Diastolic: 0

Sample values:
   systolic  diastolic
0     143.0       90.0
1     109.0       86.0
2     153.0       95.0
3     127.0       79.0
4     161.0       93.0
5     113.0       71.0
6     145.0       93.0
7     155.0       94.0
8     144.0       80.0
9     161.0      103.0

Columns remaining: ['patient_name', 'age', 'gender', 'condition', 'medication', 'visit_date', 'cholesterol', 'systolic', 'diastolic', 'visit_year', 'visit_month']


In [20]:
df

,patient_name,age,gender,condition,medication,visit_date,cholesterol,systolic,diastolic
0,donna hall,51,female,Diabetes,ATORVASTATIN,2021-06-19,207.7,143.0,90.0
1,mark young,49,male,Diabetes,METFORMIN,2020-04-06,206.3,109.0,86.0
2,lisa young,49,male,Hypertension,NONE,2018-05-23,207.5,153.0,95.0
3,daniel robinson,46,male,Asthma,ALBUTEROL,2022-10-10,203.6,127.0,79.0
4,robert garcia,49,female,Hypertension,NONE,2018-05-10,199.5,161.0,93.0
...,...,...,...,...,...,...,...,...,...
895,james wilson,58,male,Diabetes,ATORVASTATIN,2020-09-10,228.3,136.0,87.0
896,donna martinez,49,female,Heart Disease,LISINOPRIL,2023-03-28,222.7,153.0,88.0
897,emily jackson,68,male,Diabetes,ATORVASTATIN,2019-02-21,213.8,153.0,80.0
898,robert walker,19,male,Asthma,NONE,2023-04-22,170.0,121.0,70.0


In [21]:
# Cholesterol already cleaned and filled above (Fix 6)
# df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())


In [22]:
df["visit_year"] = df["visit_date"].dt.year
df["visit_month"] = df["visit_date"].dt.month

df["visit_year"] = df["visit_year"].fillna(df["visit_year"].mode()[0])
df["visit_month"] = df["visit_month"].fillna(df["visit_month"].mode()[0])



In [41]:
print("Visit year and month after extraction:")
print(df[['visit_date', 'visit_year', 'visit_month']].head(10))
print("\nMissing values:")
print("visit_year: ", df['visit_year'].isnull().sum())
print("visit_month:", df['visit_month'].isnull().sum())
print("\nUnique years:", sorted(df['visit_year'].unique()))
print("Unique months:", sorted(df['visit_month'].unique()))

Visit year and month after extraction:
  visit_date  visit_year  visit_month
0 2021-06-19        2021            6
1 2020-04-06        2020            4
2 2018-05-23        2018            5
3 2022-10-10        2022           10
4 2018-05-10        2018            5
5 2023-06-09        2023            6
6 2021-09-04        2021            9
7 2019-08-20        2019            8
8 2022-08-16        2022            8
9 2021-07-21        2021            7

Missing values:
visit_year:  0
visit_month: 0

Unique years: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
Unique months: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]


In [23]:
df.to_csv('data/cleaned_data.csv', index=False)

In [24]:
df

,patient_name,age,gender,condition,medication,visit_date,cholesterol,systolic,diastolic,visit_year,visit_month
0,donna hall,51,female,Diabetes,ATORVASTATIN,2021-06-19,207.7,143.0,90.0,2021,6
1,mark young,49,male,Diabetes,METFORMIN,2020-04-06,206.3,109.0,86.0,2020,4
2,lisa young,49,male,Hypertension,NONE,2018-05-23,207.5,153.0,95.0,2018,5
3,daniel robinson,46,male,Asthma,ALBUTEROL,2022-10-10,203.6,127.0,79.0,2022,10
4,robert garcia,49,female,Hypertension,NONE,2018-05-10,199.5,161.0,93.0,2018,5
...,...,...,...,...,...,...,...,...,...,...,...
895,james wilson,58,male,Diabetes,ATORVASTATIN,2020-09-10,228.3,136.0,87.0,2020,9
896,donna martinez,49,female,Heart Disease,LISINOPRIL,2023-03-28,222.7,153.0,88.0,2023,3
897,emily jackson,68,male,Diabetes,ATORVASTATIN,2019-02-21,213.8,153.0,80.0,2019,2
898,robert walker,19,male,Asthma,NONE,2023-04-22,170.0,121.0,70.0,2023,4
